# Week 9 Qwen2.5-7B FedLoRA suite on Kaggle T4 x2

This notebook runs the frozen `qwen7b-fedlora` development profile: FedAvg-LoRA, FFA-LoRA, FLoRA, FlexLoRA, FLoRIST, and FLoRG. It uses one independent process per T4 and never changes the scientific configuration after an OOM.

Recommended order: `preflight`, then `smoke`, then `run`. The full `all` suite contains 12 jobs (6 methods x 2 seeds); each sub-suite contains 6 jobs. Existing valid `result.json` artifacts are reused. These are development diagnostics, not a held-out thesis verdict.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
from IPython.display import FileLink, display

REPO_URL = 'https://github.com/TrgPhan/VASTLoRA.git'
REPO_REF = '1fa1b17cb5847bf42947286361147d9f13ce2ca4'
MODE = 'preflight'  # preflight / smoke / run / retry / report
METHOD_SUITE = 'all'  # all / extended / spectral
RUN_TRAINING = False
RUN_PREFLIGHT_TESTS = True
GPU_IDS = [0, 1]
MAX_JOBS = None  # limits only newly pending jobs
RESUME_ROOTS = []  # paths from attached Kaggle datasets

METHOD_SUITES = {
    'extended': ['fedavg_lora', 'ffa_lora', 'flora_lora'],
    'spectral': ['flexlora', 'florist', 'florg'],
}
METHOD_SUITES['all'] = METHOD_SUITES['extended'] + METHOD_SUITES['spectral']
if MODE not in {'preflight', 'smoke', 'run', 'retry', 'report'}:
    raise ValueError(f'Invalid MODE: {MODE}')
if METHOD_SUITE not in METHOD_SUITES:
    raise ValueError(f'Invalid METHOD_SUITE: {METHOD_SUITE}')
if MODE in {'preflight', 'report'} and RUN_TRAINING:
    raise ValueError(f'{MODE} mode never starts training')
if len(GPU_IDS) != len(set(GPU_IDS)) or any(gpu < 0 for gpu in GPU_IDS):
    raise ValueError('GPU_IDS must be distinct non-negative integers')

WORK_ROOT = Path('/kaggle/working')
RUN_NAME = 'week9_7b_fedlora_smoke_v1' if MODE == 'smoke' else 'week9_7b_fedlora_v1'
OUTPUT_ROOT = WORK_ROOT / RUN_NAME
SELECTED_METHODS = METHOD_SUITES[METHOD_SUITE]
os.environ.update(
    PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
    TOKENIZERS_PARALLELISM='false',
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
    HF_HUB_DISABLE_XET='1',
)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({'mode': MODE, 'suite': METHOD_SUITE, 'methods': SELECTED_METHODS,
       'training': RUN_TRAINING, 'output_root': str(OUTPUT_ROOT)})

## Fetch the frozen implementation

The checkout is detached at the audited runtime commit. Re-running this cell discards only changes inside the temporary Kaggle clone.

In [ ]:
REPO_DIR = WORK_ROOT / 'VASTLoRA-week9-7b'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
resolved = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(
    ['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True
).strip()
if resolved != REPO_REF or dirty:
    raise RuntimeError({'resolved': resolved, 'expected': REPO_REF, 'dirty': dirty})
RUNNER = REPO_DIR / 'scripts/run_week9_generation.py'
if not RUNNER.exists():
    raise FileNotFoundError(RUNNER)
print({'repo': str(REPO_DIR), 'commit': resolved})

## Install without replacing Kaggle CUDA PyTorch

The current PyTorch build is constrained, and an incompatible preinstalled `torchao` is removed. Restarting the kernel is normally unnecessary because PyTorch is not imported before this cell.

In [ ]:
torch_version = importlib.metadata.version('torch').split('+')[0]
constraint = WORK_ROOT / 'kaggle-torch-constraint.txt'
constraint.write_text(f'torch=={torch_version}\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--constraint', str(constraint),
    '-e', f'{REPO_DIR}[scale,dev,generation]',
    'transformers==5.10.2', 'peft==0.20.0', 'accelerate==1.14.0',
    'bitsandbytes==0.50.2', 'datasets==5.0.0', 'rouge-score==0.1.2',
], check=True)
print({name: importlib.metadata.version(name) for name in (
    'torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'datasets'
)})

## Tests, data audit, and immutable job plan

This cell validates method integration, the event schedule, retained Dolly data, and the exact commands before any model is loaded.

In [ ]:
def runner_command(action=None):
    command = [
        sys.executable, '-u', str(RUNNER),
        '--profile', 'qwen7b-fedlora',
        '--output-root', str(OUTPUT_ROOT),
    ]
    if MODE == 'smoke':
        command.append('--smoke')
    for method in SELECTED_METHODS:
        command.extend(['--method', method])
    if action:
        command.append(action)
    return command

if RUN_PREFLIGHT_TESTS:
    test_files = [
        'tests/test_spectral_fedlora.py',
        'tests/test_persistent_factor_baselines.py',
        'tests/test_fed_lora_baselines.py',
        'tests/test_week8_spectral_integration.py',
        'tests/test_week9_generation.py',
        'tests/test_week9_7b_fedlora.py',
    ]
    subprocess.run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=REPO_DIR, check=True)

subprocess.run(runner_command('--dry-run'), cwd=REPO_DIR, check=True)
subprocess.run(runner_command('--prepare-only'), cwd=REPO_DIR, check=True)
subprocess.run(runner_command('--plan-only'), cwd=REPO_DIR, check=True)
plan = json.loads((OUTPUT_ROOT / 'job_plan.json').read_text(encoding='utf-8'))
expected = len(SELECTED_METHODS) * (1 if MODE == 'smoke' else 2)
if len(plan) != expected:
    raise RuntimeError(f'Expected {expected} selected jobs, got {len(plan)}')
print({'jobs': len(plan), 'status_counts': {s: sum(j['status'] == s for j in plan)
      for s in sorted({j['status'] for j in plan})}})
display(plan[:3])

## Train or resume on T4 x2

Set `MODE='smoke'` and `RUN_TRAINING=True` first. After it succeeds, use `MODE='run'`. Use `MODE='retry'` only to archive incomplete run directories and restart those exact jobs. FLoRG has the highest memory pressure; an OOM should be moved to a larger GPU rather than changing its rank or update budget.

In [ ]:
if RUN_TRAINING:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('Enable GPU T4 x2 in Kaggle settings')
    if torch.cuda.device_count() < len(GPU_IDS):
        raise RuntimeError(f'Requested {GPU_IDS}, found {torch.cuda.device_count()} CUDA devices')
    gpu_state = []
    for gpu in GPU_IDS:
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu)
        state = {
            'id': gpu, 'name': torch.cuda.get_device_name(gpu),
            'free_GiB': round(free_bytes / 2**30, 2),
            'total_GiB': round(total_bytes / 2**30, 2),
        }
        gpu_state.append(state)
        if state['free_GiB'] < 13.0:
            raise RuntimeError(f'GPU {gpu} has insufficient free memory: {state}')
    print(gpu_state)
    subprocess.run(['nvidia-smi'], check=True)

    command = runner_command()
    for gpu in GPU_IDS:
        command.extend(['--gpu', str(gpu)])
    command.extend(['--workers-per-gpu', '1'])
    if MODE == 'retry':
        command.append('--retry-incomplete')
    if MAX_JOBS is not None:
        command.extend(['--max-jobs', str(MAX_JOBS)])
    for root in RESUME_ROOTS:
        command.extend(['--resume-root', str(Path(root))])

    print('Launching:', ' '.join(map(str, command)))
    process = subprocess.Popen(
        command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code:
        log_files = sorted((OUTPUT_ROOT / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)
        if log_files:
            print('Latest worker log:', log_files[-1])
            print(''.join(log_files[-1].read_text(errors='replace').splitlines(True)[-80:]))
        raise subprocess.CalledProcessError(return_code, command)
else:
    print('Training is disabled. Set MODE and RUN_TRAINING deliberately, then rerun from the first cell.')

## Validate and summarize completed jobs

The report validates provenance and matched data/schedule identities before reading NLL, ROUGE-L, harmful updates, runtime, and peak memory. Partial reports remain explicitly partial.

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO_DIR / 'scripts'))
import run_week9_generation as week9_runner
from week9_artifacts import validate_run

if not (OUTPUT_ROOT / 'matrix.json').exists():
    print('No matrix.json yet; run the preflight cell first.')
else:
    matrix = json.loads((OUTPUT_ROOT / 'matrix.json').read_text(encoding='utf-8'))
    rows, missing, issues = [], [], []
    paired_identity = {}
    for method, seed, config, result_path in week9_runner.specs(matrix, OUTPUT_ROOT):
        if method not in SELECTED_METHODS:
            continue
        key = f'{method}/seed{seed}'
        if not result_path.exists():
            missing.append(key)
            continue
        try:
            payload, identity = validate_run(
                result_path, config=config, method=method, seed=seed, matrix=matrix
            )
            pair_key = (config['experiment']['regime_name'], seed)
            if pair_key in paired_identity and paired_identity[pair_key] != identity:
                raise ValueError('data/schedule identity differs across paired methods')
            paired_identity[pair_key] = identity
            m = payload['metrics']
            rows.append({
                'method': method, 'seed': seed,
                'token_nll': m['final_token_nll'],
                'perplexity': m['final_perplexity'],
                'rouge_l': m['final_rouge_l'],
                'rouge_l_precision': m.get('final_rouge_l_precision'),
                'rouge_l_recall': m.get('final_rouge_l_recall'),
                'exact_match': m['final_exact_match'],
                'harmful': m['harmful_update_rate'],
                'late_harmful': m.get('late_harmful_update_rate'),
                'late_events': m['late_event_count'],
                'acceptance': m['acceptance_rate'],
                'runtime_minutes': m['runtime_seconds'] / 60.0,
                'peak_vram_gib': m['peak_cuda_memory_gib'],
                'generation_limit_rate': m['final_generation_limit_rate'],
                'git_commit': payload.get('git_commit'),
            })
        except Exception as error:
            issues.append(f'{key}: {type(error).__name__}: {error}')

    runs = pd.DataFrame(rows)
    report_dir = OUTPUT_ROOT / 'notebook_report'
    report_dir.mkdir(parents=True, exist_ok=True)
    runs.to_csv(report_dir / 'runs.csv', index=False)
    expected = len(SELECTED_METHODS) * (1 if matrix['phase'] == 'smoke' else 2)
    print({'validated': len(rows), 'expected': expected, 'missing': len(missing), 'issues': issues})
    if not runs.empty:
        summary = runs.groupby('method', as_index=False).agg(
            seeds=('seed', 'count'), token_nll=('token_nll', 'mean'),
            perplexity=('perplexity', 'mean'), rouge_l=('rouge_l', 'mean'),
            rouge_l_precision=('rouge_l_precision', 'mean'),
            rouge_l_recall=('rouge_l_recall', 'mean'),
            harmful=('harmful', 'mean'), late_harmful=('late_harmful', 'mean'),
            minimum_late_events=('late_events', 'min'), acceptance=('acceptance', 'mean'),
            runtime_minutes=('runtime_minutes', 'mean'), peak_vram_gib=('peak_vram_gib', 'max'),
            generation_limit_rate=('generation_limit_rate', 'mean'),
        ).sort_values(['token_nll', 'rouge_l'], ascending=[True, False])
        summary.to_csv(report_dir / 'summary.csv', index=False)
        report_text = '# Week 9 Qwen2.5-7B FedLoRA development summary\n\n' + summary.to_markdown(index=False)
        report_text += f'\n\nValidated: {len(rows)}/{expected}; missing: {len(missing)}; issues: {len(issues)}.\n'
        (report_dir / 'summary.md').write_text(report_text, encoding='utf-8')
        display(summary)
    if missing:
        print('Missing jobs:', missing)

## Package outputs

Download the ZIP before ending the Kaggle session, or publish the output as a private Kaggle dataset for a later resume.

In [ ]:
if OUTPUT_ROOT.exists():
    archive = Path(shutil.make_archive(
        str(WORK_ROOT / f'{OUTPUT_ROOT.name}_{METHOD_SUITE}'), 'zip',
        root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name,
    ))
    print({'archive': str(archive), 'size_GiB': round(archive.stat().st_size / 2**30, 3)})
    display(FileLink(str(archive)))
else:
    print('No output directory exists yet.')